# Exercise 4: Judging a judge

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Generation and Dialogue · version 0.1.0*

> Create pairs in which the second sentence contains one Turkic-specific error from the chapter. Ask an LLM judge which sentence is correct, with the order counterbalanced. Which error types does it miss, and does it prefer one position?

**Setup.** `pip install llama-cpp-python fasttext-wheel requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). The generator is **Qwen2.5-1.5B-Instruct** in 8-bit GGUF form, run on the CPU with llama.cpp: set `QWEN_GGUF` to the file. The book's run converted the Hugging Face checkpoint at revision `989aa79` with llama.cpp's `convert_hf_to_gguf.py --outtype q8_0` (SHA-256 recorded in the outputs); Qwen's own `Qwen2.5-1.5B-Instruct-GGUF` q8_0 file is an equivalent starting point. Language identification uses GlotLID v3 (`GLOTLID_PATH`, otherwise downloaded from the Hub), morphology the Apertium analysers that come with TurkicNLP. Decoding is greedy (temperature 0) and every model reply is cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so the outputs can be inspected and a rerun is exact. **One small model is used throughout.** Its errors illustrate the measurement methods; they are not a ranking of models or a statement about larger ones.

Correct sentences come from SIB-200 (FLORES-200) Kazakh. Each of four automatic perturbations is applied to 25 different sentences. The judge sees both sentences, in random order, and answers with a letter; the prompt is given once in English and once in Kazakh.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_test.txt"
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# ---- the generator, with every reply cached ---------------------------------------------------
_LLM = {}
REPLIES = CACHE / "ch14-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["reply"]

def chat(messages, max_tokens=200):
    """Greedy reply of Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) to a list of chat messages."""
    key = hashlib.sha1(json.dumps([messages, max_tokens], ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        if "m" not in _LLM:
            from llama_cpp import Llama
            _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
        out = _LLM["m"].create_chat_completion(messages=messages, temperature=0.0, max_tokens=max_tokens)
        _CACHED[key] = out["choices"][0]["message"]["content"].strip()
        REPLIES.parent.mkdir(parents=True, exist_ok=True)
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "messages": messages, "reply": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

# ---- automatic fidelity signals ---------------------------------------------------------------
_LID = {}
def lid(text):
    """Unrestricted GlotLID v3 label and probability of one line of text."""
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

SENT = re.compile(r"(?<=[.!?…])\s+|\n+")
def sentences(text):
    return [s.strip() for s in SENT.split(text) if len(re.findall(r"[^\W\d_]", s)) >= 3]

def script_shares(text):
    letters = [c for c in text if c.isalpha()]
    count = collections.Counter("Cyrl" if "CYRILLIC" in unicodedata.name(c, "") else
                                "Latn" if "LATIN" in unicodedata.name(c, "") else "other" for c in letters)
    return {k: v / len(letters) for k, v in count.items()} if letters else {}

WORD = re.compile(r"[^\W\d_]+")
_FST = {}
def lookup(lang):
    """Raw Apertium analyser of TurkicNLP for kaz (Cyrillic) or uzb (Latin): word -> [(reading, weight)]."""
    if lang not in _FST:
        import hfst, turkicnlp
        from turkicnlp.resources.registry import ModelRegistry
        script = {"kaz": "Cyrl", "uzb": "Latn"}[lang]
        turkicnlp.download(lang, processors=["morph"])
        path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
        fst = hfst.HfstInputStream(str(path)).read()
        _FST[lang] = (lambda w: fst.lookup(w) or fst.lookup(w.lower())), sha256(path)
    return _FST[lang][0]

def accepted(word, lang):
    return any(r and not r.startswith("*") and math.isfinite(w) for r, w in lookup(lang)(word))

def coverage(texts, lang):
    words = [w for t in texts for w in WORD.findall(t)]
    ok = sum(accepted(w, lang) for w in words)
    return ok, len(words)

print("generator:", Path(QWEN_GGUF).name, "| sha256", sha256(QWEN_GGUF)[:16] if Path(QWEN_GGUF).exists() else "missing")

In [ ]:
# One Turkic-specific error per pair, made automatically from correct SIB-200 (FLORES-200) Kazakh sentences.
import csv, io
rows = []
for split in ("train", "dev", "test"):
    rows += [r["text"] for r in csv.DictReader(io.StringIO(fetch(f"{SIB}kaz_Cyrl/{split}.tsv").read_text(encoding="utf-8")), delimiter="\t")]
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator
latin = Transliterator("kaz", Script.CYRILLIC, Script.COMMON_TURKIC)
BACK, FRONT = set("аоұы"), set("әеөүі")
def harmony(s):          # a back-vowel noun with its plural suffix in the front-vowel form (кітаптар -> кітаптер)
    for w in WORD.findall(s):
        m = re.fullmatch(r"(.+?)(лар|дар|тар)(.*)", w)
        if m and m.group(1)[-1] not in "аоұыәеөүі" and [c for c in m.group(1) if c in BACK | FRONT][-1:] and \
           [c for c in m.group(1) if c in BACK | FRONT][-1] in BACK:
            return s.replace(w, m.group(1) + m.group(2)[0] + "ер" + m.group(3), 1)
def russian(s):          # a Kazakh conjunction replaced by its Russian equivalent
    for kk, ru in (("және", "и"), ("бірақ", "но"), ("немесе", "или")):
        if re.search(rf"\b{kk}\b", s):
            return re.sub(rf"\b{kk}\b", ru, s, count=1)
def kyrgyz(s):           # a Kazakh form replaced by its Kyrgyz cognate
    for kk, ky in (("және", "жана"), ("емес", "эмес"), ("болды", "болду"), ("деп", "деп"), ("мен", "менен")):
        if kk != ky and re.search(rf"\b{kk}\b", s):
            return re.sub(rf"\b{kk}\b", ky, s, count=1)
def script(s):           # the longest word written in the Latin (CTS) alphabet
    w = max(WORD.findall(s), key=len)
    return s.replace(w, latin.transliterate(w), 1)
PERTURB = {"harmony": harmony, "Russian word": russian, "Kyrgyz word": kyrgyz, "Latin script": script}
pairs, used = [], set()
for name, fn in PERTURB.items():
    n = 0
    for i, s in enumerate(rows):
        if n == 25:
            break
        if i in used or not 8 <= len(s.split()) <= 25:
            continue
        bad = fn(s)
        if bad and bad != s:
            pairs.append((name, s, bad)); used.add(i); n += 1
print("pairs per error type:", dict(collections.Counter(p[0] for p in pairs)))
for name in PERTURB:
    ex = next(p for p in pairs if p[0] == name)
    diff = [(a, b) for a, b in zip(ex[1].split(), ex[2].split()) if a != b]
    print(f"  {name:<13} {diff}")
PROMPT = {"en": ("Which of the two Kazakh sentences is correct, well-formed Kazakh? Answer with the letter A or B only.\n\nA: {a}\nB: {b}", "Answer:"),
          "kk": ("Қай сөйлем дұрыс, сауатты қазақ тілінде жазылған? Тек A немесе B әрпімен жауап беріңіз.\n\nA: {a}\nB: {b}", "Жауап:")}
rng = random.Random(0)
order = [rng.random() < 0.5 for _ in pairs]            # True: the correct sentence is shown as A
print(f"\njudge: Qwen2.5-1.5B-Instruct; correct sentence shown first in {sum(order)} of {len(pairs)} pairs")
for lang, (template, _) in PROMPT.items():
    hits, letters = collections.Counter(), collections.Counter()
    for (name, good, bad), good_first in zip(pairs, order):
        a, b = (good, bad) if good_first else (bad, good)
        reply = chat([{"role": "user", "content": template.format(a=a, b=b)}], max_tokens=3).strip().upper()
        letter = "A" if reply.startswith("A") else "B" if reply.startswith("B") else "?"
        letters[letter] += 1
        hits[name] += letter == ("A" if good_first else "B")
    print(f"prompt in {'English' if lang == 'en' else 'Kazakh'}: answers {dict(letters)}; correct choice by error type: "
          + ", ".join(f"{k} {hits[k]}/25" for k in PERTURB) + f"; total {sum(hits.values())}/{len(pairs)}")

## Worked solution

Output of `experiments/ch14_exercises/run_ch14_exercises.py`:

```
pairs per error type: {'harmony': 25, 'Russian word': 25, 'Kyrgyz word': 25, 'Latin script': 25}
  harmony       [('қайықтар', 'қайықтер')]
  Russian word  [('және', 'и')]
  Kyrgyz word   [('емес', 'эмес')]
  Latin script  [('басталғаннан', 'bastalğannan')]

judge: Qwen2.5-1.5B-Instruct; correct sentence shown first in 37 of 100 pairs
prompt in English: answers {'A': 91, 'B': 4, '?': 5}; correct choice by error type: harmony 9/25, Russian word 11/25, Kyrgyz word 8/25, Latin script 6/25; total 34/100
prompt in Kazakh: answers {'A': 88, '?': 12}; correct choice by error type: harmony 6/25, Russian word 12/25, Kyrgyz word 9/25, Latin script 5/25; total 32/100
```

**The judge strongly favours "A" in this setup.** With English prompts it returns A 91 times, B four times and an unparsable reply five times. With Kazakh prompts it returns A 88 times and an unparsable reply twelve times. It chooses the original sentence in 34 and 32 of the 100 pairs, respectively.

The original sentence appears first in only 37 pairs: randomisation did not produce balanced order. Also, the automatic changes have not been checked by speakers. These are scores against construction labels, not verified grammaticality judgements.

Validate the pairs with speakers, present each pair in both orders and report unparsable replies separately. This would test position effects and error detection more directly. The current results do not support using this model and prompt as a reliable judge, but do not establish how other models would perform.

## Your turn — extension

1. Score the pairs by the judge's log-probability of each sentence instead of asking it, and compare.
2. Add a fifth error type, a wrong case suffix, and check the new pairs with a speaker.